In [ ]:
from glob import glob
file_path = r'C:\skn29_lecture\머신러닝\실습\data\*.csv'
files = glob(file_path)
files = files[4:]

'C:\\skn29_lecture\\머신러닝\\실습\\data\\소상공인시장진흥공단_상가(상권)정보_서울_202512.csv'

In [2]:
import pandas as pd
shop = pd.read_csv(files[8])

In [11]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from lightgbm import LGBMRegressor

# ---------------------------
# 0. 데이터 준비
# ---------------------------
# shop: 기존 데이터프레임
shop = shop.dropna(subset=['위도','경도','상권업종중분류명']).reset_index(drop=True)

# 좌표 → 라디안 변환
coords = np.radians(shop[['위도','경도']].values)

# BallTree 생성
tree = BallTree(coords, metric='haversine')

# 반경 설정 (500m)
radius = 0.5 / 6371


# ---------------------------
# 1. 전체 점포 밀집도
# ---------------------------
store_counts = tree.query_radius(coords, r=radius, count_only=True)
shop['store_density_500m'] = store_counts


# ---------------------------
# 2. 동일 업종 밀집도 (예: 카페)
# ---------------------------
target_category = '카페'

cafe_mask = (shop['상권업종소분류명'] == target_category)
cafe_coords = np.radians(shop.loc[cafe_mask, ['위도','경도']].values)
cafe_coords

array([[0.65560758, 2.21326064],
       [0.65642665, 2.21511409],
       [0.65643182, 2.21532579],
       ...,
       [0.65618254, 2.21593513],
       [0.65442198, 2.21700828],
       [0.65528494, 2.21526783]], shape=(21624, 2))

In [ ]:
cafe_tree = BallTree(cafe_coords, metric='haversine')

cafe_counts = cafe_tree.query_radius(coords, r=radius, count_only=True)
shop['cafe_density_500m'] = cafe_counts


# ---------------------------
# 3. 경쟁 강도
# ---------------------------
shop['competition_ratio'] = (
    shop['cafe_density_500m'] / shop['store_density_500m']
).replace([np.inf, -np.inf], 0).fillna(0)


# ---------------------------
# 4. 상권 다양성
# ---------------------------
indices = tree.query_radius(coords, r=radius)

diversity = []
for idx_list in indices:
    categories = shop.iloc[idx_list]['상권업종중분류명']
    diversity.append(categories.nunique())

shop['category_diversity_500m'] = diversity


# ---------------------------
# 5. Feature 정리
# ---------------------------
features = [
    'store_density_500m',
    'cafe_density_500m',
    'competition_ratio',
    'category_diversity_500m',
    '위도',
    '경도'
]

X = shop[features]


# ---------------------------
# 6. Score 생성 (라벨 대신)
# ---------------------------
shop['score'] = (
    0.4 * shop['store_density_500m']
    - 0.3 * shop['competition_ratio']
    + 0.3 * shop['category_diversity_500m']
)

y = shop['score']

In [ ]:
# ---------------------------
# 7. 모델 학습
# ---------------------------
model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42
)

model.fit(X, y)


# ---------------------------
# 8. 후보 위치 (Grid 생성)
# ---------------------------
lat_min, lat_max = shop['위도'].min(), shop['위도'].max()
lon_min, lon_max = shop['경도'].min(), shop['경도'].max()

lat_grid = np.linspace(lat_min, lat_max, 50)
lon_grid = np.linspace(lon_min, lon_max, 50)

grid = np.array(np.meshgrid(lat_grid, lon_grid)).T.reshape(-1, 2)

grid_df = pd.DataFrame(grid, columns=['위도','경도'])

grid_coords = np.radians(grid)


# ---------------------------
# 9. Grid Feature 생성
# ---------------------------
# 전체 밀집도
grid_store_counts = tree.query_radius(grid_coords, r=radius, count_only=True)

# 카페 밀집도
grid_cafe_counts = cafe_tree.query_radius(grid_coords, r=radius, count_only=True)

grid_df['store_density_500m'] = grid_store_counts
grid_df['cafe_density_500m'] = grid_cafe_counts

# 경쟁 강도
grid_df['competition_ratio'] = (
    grid_df['cafe_density_500m'] / grid_df['store_density_500m']
).replace([np.inf, -np.inf], 0).fillna(0)

# 다양성 (주의: 느림)
grid_indices = tree.query_radius(grid_coords, r=radius)

grid_diversity = []
for idx_list in grid_indices:
    categories = shop.iloc[idx_list]['상권업종중분류명']
    grid_diversity.append(categories.nunique())

grid_df['category_diversity_500m'] = grid_diversity


# ---------------------------
# 10. 예측 및 추천
# ---------------------------
grid_X = grid_df[features]

grid_df['pred_score'] = model.predict(grid_X)

# Top 10 추천
top_k = grid_df.sort_values('pred_score', ascending=False).head(10)

print(top_k)